<a target="_blank" href="https://colab.research.google.com/github/AndreiSokolovskii/hackaton_december_2025/blob/main/Hackaton_december_2025_w_AlphaFold_v2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Notebook from Hackaton December 2025 for the model**

**Introduction.**
*   The notebook is still in development.
*   At this moment, the following  prediction regimes are available:

--------------------------------------------

**3D Protein backbone structure ⇒ Sequence**
1. full sequence prediction from backbone blueprint
2. partiall sequence design from partially defined structure


**Important notice**
* WIP





In [ ]:
%%capture
#@title #Installation required libraries, downloading model weights.
# #@markdown ESMfold could be used for designed sequnces foldability rapid test.
# #@markdown ---
# #@markdown ***only in case of single strtucture prediction***.

# #@markdown The code ESMfold implementation was taken from Sergey Ovchinnikov [GitHub_ColabFold](https://github.com/sokrypton/ColabFold/)
# #@markdown ---
# #@markdown
#@markdown This step can take up to ~1 mins (in case of CPU colab enviroment ~5 min).
#@markdown
#@markdown With installation of ESMfold ~7 mins.

import os, re


if not os.path.isfile("ENV_READY"):
  # !pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 torch_geometric
  !pip install torch_geometric
  !pip install pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
  # !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
  !pip install biopython pydssp
  !git clone https://github.com/AndreiSokolovskii/hackaton_december_2025.git
  !cp hackaton_december_2025/*.py .
  os.system('touch ENV_READY')

from colab_upload_helper import process_upload, create_output_folder,_read_meta


ESM_fold_for_test = False #@param {type:"boolean"}
version = "1"
model_name = "esmfold.model"

if ESM_fold_for_test:
  import os, time
  if not os.path.isfile(model_name):
    # download esmfold params
    os.system("apt-get install aria2 -qq")
    os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

    if not os.path.isfile("finished_install"):
      # install libs
      print("installing libs...")
      os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif")
      os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")

      print("installing openfold...")
      # install openfold
      os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git")

      print("installing esmfold...")
      # install esmfold
      os.system(f"pip install -q git+https://github.com/sokrypton/esm.git")
      os.system("touch finished_install")

    # wait for Params to finish downloading...
    while not os.path.isfile(model_name):
      time.sleep(5)
    if os.path.isfile(f"{model_name}.aria2"):
      print("downloading params...")
    while os.path.isfile(f"{model_name}.aria2"):
      time.sleep(5)

**Instructions**
---
---

The use of `input` can be done as follows:
- `input = '6X9Z'` - sequence design for the protein backbone downloaded from RCSB
- `input = ''` - allows to upload the structure from local storage in format  ***.pdb*** or ***.zip*** with a lot of PDBs.
---
- `use_last = 'True / False'` - the last time uploaded file will be used for prediction to skip uploading.
---
- `design_position = 'all'` - design all residues in structure.
- `design_position = '11 12 14:18'` - designing position lists specified in python style.
---
- `regime = 'One Shot Fast'` - fast design regime with generating all sequences at once per structure.
- `regime = 'One Shot Diverse'` - slower regime with generating more diverse set of sequences, ***probably***, should be tested.
- `regime = 'Iterative sampling'` - the slowest regime with iterative generating sequences **inspired by ProteinMPNN behaviour**.
- `regime = 'Iterative Refinement'` - the New Regime with iterative refinement sequences.
---

In [ ]:
#@markdown ##Settings and run.
input = '6X9Z' #@param {type:"string"}
use_last = False #@param {type:"boolean"}
chains = "A" #@param {type:"string"}
homooligomer = False #@param {type:"boolean"}
chains = re.sub("[^A-Za-z]+",",", chains)


design_position = 'all'#@param {type:"string"}
#@markdown - Position lists, e.g. 11 12 14:18. Default = all => design all residues

regime = "Iterative sampling" #@param ["One Shot Fast", "One Shot Diverse", "Iterative sampling", "Iterative Refinement"]

path_list = process_upload(is_same=use_last, pdb_code=input)

model_version = "v2" #@param ["v1", "v2"]
#@markdown - `model_version = v1` - trained in fully-blind regime
#@markdown - `model_version = v2` - trained with 10% masking regime. More preferable for partial redesign

num_seq_per_target = 11 # @param {"type":"raw"}
#@markdown - `num_seq_per_target = '7'` -  7 sequences will be generated per structure.

sampling_temp = "0.7" #@param ["0.01", "0.1", "0.15", "0.2", "0.5", "0.7", "1", "1.5", "2"]
#@markdown - Sampling temperature T=0.0 means taking argmax, T \>> 1.0 means sample randomly.
rm_aa = "" #@param {type:"string"}
#@markdown - `rm_aa='C'` - do not use [C]ysteines.
progress_bar = True #@param {type:"boolean"}

show_generated_fasta = False #@param {type:"boolean"}

#@markdown ----
experimental_batch_generation = False #@param {type:"boolean"}
#@markdown - will work in case of large structure set


from run import main as run

class colab_args(object):
  def __init__(self, regime, model_version, num_seq_per_target,
               sampling_temp, rm_aa, progress_bar, design_position):
    short_regime_names = {"One Shot Fast":'OSF', "One Shot Diverse": 'OSD', "Iterative sampling":'IS', 'Iterative Refinement':'IR'}

    self.model_masked = True if model_version == 'v1' else False
    self.num_seq_per_target = int(num_seq_per_target)
    self.temperature = float(sampling_temp)
    self.suppress_AAs = [aa for aa in rm_aa] if len(rm_aa) > 0 else ['X']
    self.verbose = progress_bar
    self.design_position = str(design_position) if design_position != 'all' else '0'
    self.iterative_sampling = True if regime == 'Iterative sampling' else False
    self.one_shot_diverse = True if regime == 'One Shot Diverse' else False
    self.iterative_refin = True if regime == 'Iterative Refinement' else False
    meta = _read_meta()
    self.input_pdb = str(path_list)
    if meta['last'][-4:] == '.pdb':
      output_path = meta['last'][:-4]
    else:
      self.input_pdb = self.input_pdb + '/*.pdb'
      output_path = meta['last']

    output_path = output_path + '_r'\
     + short_regime_names[regime] + '_s'\
     + str(self.num_seq_per_target) + '_t' \
     + str(self.temperature) + '_m_' + str(model_version)
    self.output_path = str(create_output_folder(output_path))

    self.seed = 42
    self.path_to_model = 'hackaton_december_2025/weights'
    self.batched = experimental_batch_generation


args = colab_args(regime=regime, model_version=model_version,
                  num_seq_per_target=num_seq_per_target, sampling_temp=sampling_temp,
                  rm_aa=rm_aa, progress_bar=progress_bar, design_position=design_position,
                  )
run(args)

if show_generated_fasta:
  name = args.output_path+ '/'+ args.input_pdb.split('/')[-1].split('.')[0] + '.fa'
  with open(name, 'r') as fin:
    for line in fin.readlines():
      print(line.strip())
      if not line.startswith('>'):
        print()
import pandas as pd
from google.colab import data_table
data_table.enable_dataframe_formatter()
labels = ["score","seqid","seq"]
name = args.output_path+ '/'+ args.input_pdb.split('/')[-1].split('.')[0] + '.fa'
data=dict()
with open(name, 'r') as fin:
    for line in fin:
      if line.startswith('>'):
        val = line.strip().split(";")
        if not val[0].split(" ")[-1].startswith(">"):
          des_id = int(val[0].split(" ")[0].split("_")[-1])
          seq_id = float(val[0].split(" ")[-1])
          score = float(val[1].split(" ")[-1])
          seq = next(fin)
          data[des_id] = [score, seq_id, seq.strip()]
df = pd.DataFrame.from_dict(data, orient='index', columns=labels)

df_out = os.path.join(args.output_path, 'the_model_results.csv')
df.to_csv(df_out)
data_table.DataTable(df.round(3))

In [ ]:

#@title Run AlphaFold Prediction on designed sequences (optional)
#@markdown ###AlphaFold Options

num_recycles = 3 #@param ["0","1","2","3","6","12","24","48"] {type:"raw"}
num_models = 5 #@param ["1","2","3","4","5"] {type:"raw"}
use_multimer = False
use_templates = False #
rm_template_interchain = False
import os
try:
  import colabdesign
except:
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
  os.system("ln -s /usr/local/lib/python3.12/dist-packages/colabdesign colabdesign")
  !cp hackaton_december_2025/patches/*.patch /usr/local/lib/python3.12/dist-packages/colabdesign/af/ ; cd /usr/local/lib/python3.12/dist-packages/colabdesign/af/; patch model.py  < model.patch; patch design.py  < design.patch; patch utils.py  < utils.patch
if not os.path.isdir("params"):
  os.system("mkdir params")
  os.system("apt-get install aria2 -qq")
  os.system("aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
  os.system("tar -xf alphafold_params_2022-12-06.tar -C params")
if not os.path.isdir(os.path.join(args.output_path, 'all_pdb')):
  os.system(f"mkdir {os.path.join(args.output_path, 'all_pdb')}")
else:
  os.system(f"rm {os.path.join(args.output_path, 'all_pdb')}/*")

from colabdesign.af import mk_af_model
import tqdm.notebook
TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'


af_model = mk_af_model(use_multimer=False,
                         use_templates=False,
                         best_metric="plddt", initial_guess=False)
prep_flags = {"chain":",".join(chains),
                  "copies":1,
                  "homooligomer":homooligomer,
                  "rm_aa":rm_aa,
                  "rm_template_seq": True}
af_terms = ["plddt","ptm","pae","rmsd", 'dgram_cce', 'i_ptm', 'composite']
labels_af = ["seqid","score", 'sequence'] + af_terms
out = {}
af_model.set_seed(0)

data_af = []

pdb_filename =  args.output_path+ '/'+ args.input_pdb.split('/')[-1].split('.')[0] + '_polyGly.pdb'
print("running AlphaFold...")
with tqdm.notebook.tqdm(total=df.shape[0], bar_format=TQDM_BAR_FORMAT) as pbar:
  af_model.prep_inputs(pdb_filename)
  for k in af_terms: out[k] = []
  for n,row in df.iterrows():
    seq = row[2]
    af_model.predict(seq=seq, num_recycles=num_recycles, verbose=False, num_models=num_models, seed=42,dropout=False)
    all_plddt = af_model.aux["all"]["plddt"].mean(axis=-1)
    best_model_idx = all_plddt.argmax()
    (rmsd, ptm, plddt) = (af_model.aux["log"][k] for k in ["rmsd","ptm","plddt"])
    af_model.aux["log"]["composite"] = ptm * plddt
    af_model._save_results(save_best=True, verbose=False)
    af_model.save_current_pdb(f"{args.output_path}/all_pdb/n{n}.pdb", best_idx=best_model_idx)
    af_model._k += 1
    out = {t:af_model.aux["log"][t] for t in af_terms}
    out['seqid'] = float(row[1])
    out['score'] = float(row[0])
    out['sequence'] = seq
    data_af.append(out)

    pbar.update(1)
daf = pd.DataFrame(data_af, columns=labels_af)
daf.to_csv(os.path.join(args.output_path, 'alphafold_results.csv'))
data_table.DataTable(daf.sort_values("plddt").round(3))

In [ ]:
#@title display the best from AlphaFold (optional) {run: "auto"}



show_idx = 8 #@param {type:"integer"}
#@markdown - Enter index of protein to show, if `show_best` is disabled.
#@markdown - Note: these are NOT sorted and correspond to
#@markdown the index in pandas dataframe above.

color = "confidence" #@param ["confidence", "rainbow", "chain"]
if color == "confidence": color = "pLDDT"

show_sidechains = False #@param {type:"boolean"}
show_mainchains = True #@param {type:"boolean"}
color_HP = False #@param {type:"boolean"}

from colabdesign.shared.protein import _np_get_cb, pdb_to_string

pdb_str = pdb_to_string(f"{args.output_path}/all_pdb/n{show_idx}.pdb")

af_model.plot_pdb(show_sidechains=show_sidechains,
                  show_mainchains=show_mainchains,
                  color=color, color_HP=color_HP,
                  animate=False, pdb_str=pdb_str)

#show_pdb(best_pdb_str, color=color,
#         show_sidechains=show_sidechains,
#         show_mainchains=show_mainchains,
#         Ls=[len(seqSS[best_ind])]).show()

In [ ]:
#@title ESMfold foldability rapid test was tested ***only in case of single strtucture prediction***.
from string import ascii_uppercase, ascii_lowercase
import hashlib, re, os
import numpy as np
import torch
from jax.tree_util import tree_map
import matplotlib.pyplot as plt
from scipy.special import softmax
import gc
model = torch.load(model_name, weights_only=False)
model.eval().cuda().requires_grad_(False)
def parse_output(output):
  pae = (output["aligned_confidence_probs"][0] * np.arange(64)).mean(-1) * 31
  plddt = output["plddt"][0,:,1]

  bins = np.append(0,np.linspace(2.3125,21.6875,63))
  sm_contacts = softmax(output["distogram_logits"],-1)[0]
  sm_contacts = sm_contacts[...,bins<8].sum(-1)
  xyz = output["positions"][-1,0,:,1]
  mask = output["atom37_atom_exists"][0,:,1] == 1
  o = {"pae":pae[mask,:][:,mask],
       "plddt":plddt[mask],
       "sm_contacts":sm_contacts[mask,:][:,mask],
       "xyz":xyz[mask]}
  return o
def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
alphabet_list = list(ascii_uppercase+ascii_lowercase)
num_recycles = 24 #@param ["0", "1", "2", "3", "6", "12", "24"] {type:"raw"}
@torch.no_grad()
def get_one_seq_prediction(sequence, num_recycles):
  jobname = args.output_path.split('/')[-1]
  jobname = re.sub(r'\W+', '', jobname)[:50]

  sequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())
  sequence = re.sub(":+",":",sequence)
  sequence = re.sub("^[:]+","",sequence)
  sequence = re.sub("[:]+$","",sequence)
  copies = 1
  sequence = ":".join([sequence] * copies)
  chain_linker = 25
  ID = jobname+"_"+get_hash(sequence)[:5]
  seqs = sequence.split(":")
  lengths = [len(s) for s in seqs]
  length = sum(lengths)
  if length > 700:
    model.set_chunk_size(64)
  else:
    model.set_chunk_size(128)

  output = model.infer(sequence,
                     num_recycles=num_recycles,
                     chain_linker="X"*chain_linker,
                     residue_index_offset=512)
  pdb_str = model.output_to_pdb(output)[0]
  output = tree_map(lambda x: x.cpu().numpy(), output)
  ptm = output["ptm"][0]
  plddt = output["plddt"][0,:,1].mean()
  return pdb_str, ptm, plddt, parse_output(output), ID

esm_output_path = os.path.join(args.output_path, 'ESM_output')
if not os.path.exists(esm_output_path):
  os.makedirs(esm_output_path)

import glob
file = glob.glob(args.output_path + '/*.fa')[0]
seqSS = []
with open(file, 'r') as fin:
  fin.readline()
  fin.readline()
  for line in fin.readlines():
    if line.startswith('>'):
      continue
    else:
      seqSS += [line.strip()]

traj = []
best_pdb_str = None
best_ptm = 0.0
best_output = None
best_ID = None
best_ind = None
prefix_s = []
for ind, sequence in enumerate(seqSS):
  torch.cuda.empty_cache()
  pdb_str, ptm, plddt, O, ID = get_one_seq_prediction(sequence, num_recycles)

  traj.append(O)
  ID = str(ind) + ';' + ID
  ID = os.path.join(esm_output_path, ID)
  print(f'seq {ind}. \t ptm: {ptm:.3f} plddt: {plddt:.1f}')
  if not os.path.exists(ID):
    os.makedirs(ID)
  prefix = f"{ID}/ptm{ptm:.3f}_r{num_recycles}_default"
  np.savetxt(f"{prefix}.pae.txt",O["pae"],"%.3f")
  with open(f"{prefix}.pdb","w") as out:
    out.write(pdb_str)
  prefix_s += [prefix]


  if ptm > best_ptm:
    best_pdb_str = pdb_str
    best_ptm = ptm
    best_output = O
    best_ID = ID
    best_ind = ind



In [ ]:
#@title display the best from ESMfold (optional) {run: "auto"}
import py3Dmol
pymol_color_list = ["#33ff33","#00ffff","#ff33cc","#ffff00","#ff9999","#e5e5e5","#7f7fff","#ff7f00",
                    "#7fff7f","#199999","#ff007f","#ffdd5e","#8c3f99","#b2b2b2","#007fff","#c4b200",
                    "#8cb266","#00bfbf","#b27f7f","#fcd1a5","#ff7f7f","#ffbfdd","#7fffff","#ffff7f",
                    "#00ff7f","#337fcc","#d8337f","#bfff3f","#ff7fff","#d8d8ff","#3fffbf","#b78c4c",
                    "#339933","#66b2b2","#ba8c84","#84bf00","#b24c66","#7f7f7f","#3f3fa5","#a5512b"]

def show_pdb(pdb_str, show_sidechains=False, show_mainchains=False,
             color="pLDDT", chains=None, vmin=50, vmax=90,
             size=(800,480), hbondCutoff=4.0,
             Ls=None,
             animate=False):

  if chains is None:
    chains = 1 if Ls is None else len(Ls)
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=size[0], height=size[1])
  if animate:
    view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  else:
    view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  if color == "pLDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':vmin,'max':vmax}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                  {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  view.zoomTo()
  if animate: view.animate()
  return view

color = "confidence" #@param ["confidence", "rainbow", "chain"]
if color == "confidence": color = "pLDDT"
show_sidechains = False #@param {type:"boolean"}
show_mainchains = True #@param {type:"boolean"}
show_pdb(best_pdb_str, color=color,
         show_sidechains=show_sidechains,
         show_mainchains=show_mainchains,
         Ls=[len(seqSS[best_ind])]).show()